# M3 data audit — SCKN × NKC field coverage & ISBN intersection

Answers three questions before M4 (matching spike):

1. **SCKN** — unique ISBNs, missing ISBNs, bestseller-frequency distribution
2. **NKC** — ISBN / OCLC / original-title coverage overall and broken down by source language
3. **Intersection** — how many SCKN ISBNs land in NKC, and what falls out

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SCKN_PATH = REPO / "data" / "raw" / "sckn_charts.csv"
NKC_PATH  = REPO / "data" / "interim" / "nkc_translations.csv"

sckn = pd.read_csv(SCKN_PATH, dtype=str).fillna("")
nkc  = pd.read_csv(NKC_PATH,  dtype=str).fillna("")

print(f"SCKN rows : {len(sckn):>8,}")
print(f"NKC rows  : {len(nkc):>8,}")

---
## 1  SCKN audit

In [ ]:
sckn_has_isbn = sckn["isbn"].str.strip() != ""
n_total       = len(sckn)
n_with_isbn   = sckn_has_isbn.sum()
n_missing_isbn = n_total - n_with_isbn

print("=== SCKN ISBN presence ===")
print(f"Total chart entries   : {n_total:>7,}")
print(f"With ISBN             : {n_with_isbn:>7,}  ({n_with_isbn/n_total:.1%})")
print(f"Missing ISBN          : {n_missing_isbn:>7,}  ({n_missing_isbn/n_total:.1%})")

sckn_isbns = sckn.loc[sckn_has_isbn, "isbn"].str.strip().str.upper()
n_unique = sckn_isbns.nunique()
print(f"\nUnique ISBNs (raw)    : {n_unique:>7,}")

In [ ]:
appearances = (
    sckn[sckn_has_isbn]
    .assign(isbn_norm=sckn.loc[sckn_has_isbn, "isbn"].str.strip().str.upper())
    .groupby("isbn_norm")
    .size()
    .rename("chart_appearances")
)

freq_dist = appearances.value_counts().sort_index()

print("=== Bestseller frequency distribution ===")
print("appearances  |  #ISBNs  |  % of unique ISBNs")
print("-" * 44)
for k, v in freq_dist.items():
    pct = v / n_unique
    tag = " ← 1×" if k == 1 else (" ← 2×" if k == 2 else "")
    if k <= 10 or k > freq_dist.index.max() - 3:
        print(f"  {k:>4}          {v:>6,}     {pct:>6.1%}{tag}")
    elif k == 11:
        print("   ...")

print(f"\nMedian appearances : {appearances.median():.0f}")
print(f"Max appearances    : {appearances.max()}  (isbn={appearances.idxmax()})")

In [ ]:
cap = 15
plot_data = freq_dist[freq_dist.index <= cap]
rest = freq_dist[freq_dist.index > cap].sum()

labels = [str(k) for k in plot_data.index] + ([f">{cap}"] if rest else [])
values = list(plot_data.values) + ([rest] if rest else [])

fig, ax = plt.subplots(figsize=(10, 3))
ax.bar(labels, values, color="steelblue")
ax.set_xlabel("Times in top-10 chart (per ISBN)")
ax.set_ylabel("# unique ISBNs")
ax.set_title("SCKN — bestseller frequency distribution")
plt.tight_layout()
plt.show()

---
## 2  NKC field coverage

In [ ]:
def coverage(series, label):
    n = (series.str.strip() != "").sum()
    return {"field": label, "present": n, "missing": len(series) - n, "coverage": n / len(series)}

rows = [
    coverage(nkc["czech_isbn"],     "ISBN (020 $a)"),
    coverage(nkc["oclc"],           "OCLC (035 $a)"),
    coverage(nkc["original_title"], "Original title (240 $a)"),
    coverage(nkc["author"],         "Author (100 $a)"),
    coverage(nkc["czech_pub_year"], "Czech pub year"),
    coverage(nkc["genres"],         "Genre (655 $a)"),
]

cov = pd.DataFrame(rows).set_index("field")
print("=== NKC field coverage (all 225 k translations) ===")
print(cov.to_string(formatters={"present": "{:,}".format, "missing": "{:,}".format, "coverage": "{:.1%}".format}))

In [ ]:
top_langs = nkc["source_lang"].value_counts().head(15).index.tolist()

rows_lang = []
for lang in top_langs:
    sub = nkc[nkc["source_lang"] == lang]
    rows_lang.append({
        "source_lang":    lang,
        "n_records":      len(sub),
        "isbn_cov":       (sub["czech_isbn"].str.strip()    != "").sum() / len(sub),
        "oclc_cov":       (sub["oclc"].str.strip()          != "").sum() / len(sub),
        "orig_title_cov": (sub["original_title"].str.strip() != "").sum() / len(sub),
    })

lang_df = pd.DataFrame(rows_lang).set_index("source_lang")
print("=== NKC coverage by source language (top 15) ===")
print(
    lang_df.to_string(
        formatters={
            "n_records":      "{:>7,}".format,
            "isbn_cov":       "{:.1%}".format,
            "oclc_cov":       "{:.1%}".format,
            "orig_title_cov": "{:.1%}".format,
        }
    )
)

In [ ]:
import numpy as np

heat_cols   = ["isbn_cov", "oclc_cov", "orig_title_cov"]
heat_labels = ["ISBN", "OCLC", "Original title"]
mat = lang_df[heat_cols].values

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(mat, aspect="auto", cmap="RdYlGn", vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label="coverage")
ax.set_xticks(range(len(heat_labels)))
ax.set_xticklabels(heat_labels)
ax.set_yticks(range(len(lang_df)))
ax.set_yticklabels([f"{lang}  (n={lang_df.loc[lang,'n_records']:,})" for lang in lang_df.index])
ax.set_title("NKC field coverage by source language")
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        ax.text(j, i, f"{mat[i, j]:.0%}", ha="center", va="center", fontsize=8,
                color="black" if 0.25 < mat[i, j] < 0.85 else "white")
plt.tight_layout()
plt.show()

---
## 3  Intersection SCKN ↔ NKC via ISBN

Normalisation: strip dashes/spaces, uppercase. NKC pipe-separated multi-ISBN values are exploded so each variant is a separate key. SCKN 12-digit strings also try `9`-prefixed form (handles ISBN-13s where SCKN dropped the leading digit).

In [ ]:
def norm_isbn(s: pd.Series) -> pd.Series:
    return s.str.replace(r"[\s\-]", "", regex=True).str.upper().str.strip()

# SCKN side
sckn_with = sckn[sckn_has_isbn].copy()
sckn_with["isbn_norm"] = norm_isbn(sckn_with["isbn"])
sckn_unique_norm = sckn_with["isbn_norm"].unique()

# NKC side — explode pipe-separated multi-ISBN values before normalising
nkc_isbn_raw = nkc[nkc["czech_isbn"].str.strip() != ""][["nkc_id", "czech_isbn"]].copy()
nkc_isbn_exploded = (
    nkc_isbn_raw
    .assign(isbn_part=nkc_isbn_raw["czech_isbn"].str.split("|"))
    .explode("isbn_part")
)
nkc_isbn_exploded["isbn_norm"] = norm_isbn(nkc_isbn_exploded["isbn_part"])
nkc_isbn_set = set(nkc_isbn_exploded["isbn_norm"].str.strip())
nkc_isbn_set.discard("")

print(f"SCKN unique normalised ISBNs : {len(sckn_unique_norm):>6,}")
print(f"NKC  unique normalised ISBNs : {len(nkc_isbn_set):>6,}")

In [ ]:
def in_nkc(isbn: str) -> bool:
    if isbn in nkc_isbn_set:
        return True
    # 12-digit strings: try prepending 9 (SCKN sometimes drops leading digit of ISBN-13)
    if len(isbn) == 12 and isbn.isdigit():
        return ("9" + isbn) in nkc_isbn_set
    return False

sckn_norm_series = pd.Series(sckn_unique_norm)
found_mask = sckn_norm_series.apply(in_nkc)
found  = sckn_norm_series[found_mask]
missed = sckn_norm_series[~found_mask]

n_found  = len(found)
n_missed = len(missed)
n_sckn   = len(sckn_unique_norm)

print("=== SCKN → NKC ISBN match ===")
print(f"SCKN unique ISBNs (normalised) : {n_sckn:>6,}")
print(f"  Found in NKC                 : {n_found:>6,}  ({n_found/n_sckn:.1%})")
print(f"  Not found in NKC             : {n_missed:>6,}  ({n_missed/n_sckn:.1%})")

In [ ]:
# Diagnose the misses
sckn_missed_rows = sckn_with[sckn_with["isbn_norm"].isin(missed)]

print(f"SCKN chart entries behind missed ISBNs: {len(sckn_missed_rows):,}")
print()

print("=== Missed ISBNs — category breakdown ===")
miss_by_cat = (
    sckn_missed_rows.drop_duplicates("isbn_norm")
    .groupby("category")
    .size()
    .sort_values(ascending=False)
)
print(miss_by_cat.to_string())
print()

print("=== Missed ISBNs — by year (first appearance) ===")
miss_first_year = (
    sckn_missed_rows
    .assign(year=sckn_missed_rows["year"].astype(int))
    .groupby("isbn_norm")["year"].min().reset_index()
    .groupby("year").size().rename("missed_isbns")
)
found_first_year = (
    sckn_with[sckn_with["isbn_norm"].isin(found)]
    .assign(year=sckn_with["year"].astype(int))
    .groupby("isbn_norm")["year"].min().reset_index()
    .groupby("year").size().rename("found_isbns")
)
year_compare = pd.concat([found_first_year, miss_first_year], axis=1).fillna(0).astype(int)
year_compare["miss_rate"] = year_compare["missed_isbns"] / (
    year_compare["found_isbns"] + year_compare["missed_isbns"]
)
print(year_compare.to_string(formatters={"miss_rate": "{:.0%}".format}))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
ax.bar(year_compare.index, year_compare["miss_rate"], color="tomato")
ax.set_xlabel("Year of first SCKN appearance")
ax.set_ylabel("Miss rate (ISBN not in NKC)")
ax.set_title("SCKN → NKC ISBN miss rate by year")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
plt.tight_layout()
plt.show()

In [ ]:
print("=== Sample of 20 missed SCKN ISBNs ===")
sample_cols = ["year", "week", "category", "rank", "isbn", "author", "title"]
print(
    sckn_missed_rows
    .drop_duplicates("isbn_norm")
    .sort_values(["year", "week"])
    .head(20)[sample_cols]
    .to_string(index=False)
)

---
## Summary

| Question | Result |
|---|---|
| SCKN unique ISBNs | see §1 |
| SCKN missing ISBN | see §1 |
| SCKN × NKC hit rate | see §3 |
| NKC ISBN coverage | see §2 heatmap |
| NKC OCLC coverage | see §2 heatmap |
| NKC original_title by lang | see §2 heatmap — eng/swe/nor best; rus worst |

**Key implications for M4:**
- Primary key: ISBN (normalise + explode pipe variants + leading-9 repair)
- Remaining misses after ISBN join are mostly Czech-original books — expected, not a data quality problem
- OCLC is the reliable fallback for Russian (91% coverage vs 10% ISBN)
- `original_title` too sparse (26% overall) to use as a primary Goodreads key; useful only as tiebreaker